In [1]:
import os
import json
import random
import copy
from collections import defaultdict
from sklearn.model_selection import train_test_split

INPUT_DIR = '/kaggle/input/datasets/allyciajoanmicheline/dataset-4class/dataset-4class'
JSON_FILE = os.path.join(INPUT_DIR, 'adas_caption_train.json')

OUTPUT_DATA_DIR = '/kaggle/working/data'
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

with open(JSON_FILE, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

fixed_data = []
for item in raw_data:
    if "image" in item:
        rel_path = item.pop("image")
        abs_path = os.path.join(INPUT_DIR, rel_path)
        item["images"] = [abs_path]
    fixed_data.append(item)

print(f"Berhasil menormalisasi {len(fixed_data)} sampel data.")

SEED = 42
random.seed(SEED)

train_data, temp_data = train_test_split(fixed_data, test_size=0.30, random_state=SEED)
val_data, test_data = train_test_split(temp_data, test_size=(1/3), random_state=SEED)

print(f"Distribusi Natural -> Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

for split_name, split_data, fname in [("Val", val_data, "qwen35_val.json"), ("Test", test_data, "qwen35_test.json")]:
    out_path = os.path.join(OUTPUT_DATA_DIR, fname)
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(split_data, f, ensure_ascii=False, indent=2)
    print(f"Data {split_name} tersimpan di: {out_path}")

class_groups = defaultdict(list)
for item in train_data:
    caption = ""
    for conv in item.get('conversations', []):
        if conv.get('from') in ['gpt', 'assistant']:
            caption = conv.get('value', '').lower()
            break
            
    if 'lubang' in caption or 'pothole' in caption:
        class_groups['pothole'].append(item)
    elif 'memanjang' in caption or 'longitudinal' in caption:
        class_groups['longitudinal_crack'].append(item)
    elif 'melintang' in caption or 'latitude' in caption:
        class_groups['latitude_crack'].append(item)
    elif 'buaya' in caption or 'alligator' in caption:
        class_groups['alligator_crack'].append(item)
    else:
        class_groups['other'].append(item)

target_samples = max([len(s) for k, s in class_groups.items() if k != 'other'], default=0)
print(f"\nTarget pemerataan ditetapkan pada: {target_samples} sampel per kelas.")

balanced_train_data = []
for kelas, sampel in class_groups.items():
    if kelas == 'other' or len(sampel) == 0:
        continue
    
    jumlah_saat_ini = len(sampel)
    sampled_items = copy.deepcopy(sampel)
    kekurangan = target_samples - jumlah_saat_ini
    
    if kekurangan > 0:
        sampled_items.extend(random.choices(sampel, k=kekurangan))
        
    balanced_train_data.extend(sampled_items)

random.shuffle(balanced_train_data)

train_out_path = os.path.join(OUTPUT_DATA_DIR, 'qwen35_train.json')
with open(train_out_path, 'w', encoding='utf-8') as f:
    json.dump(balanced_train_data, f, ensure_ascii=False, indent=2)

print(f"Data Train Seimbang: {len(balanced_train_data)} sampel tersimpan di {train_out_path}")

Berhasil menormalisasi 16710 sampel data.
Distribusi Natural -> Train: 11697 | Val: 3342 | Test: 1671
Data Val tersimpan di: /kaggle/working/data/qwen35_val.json
Data Test tersimpan di: /kaggle/working/data/qwen35_test.json

Target pemerataan ditetapkan pada: 4396 sampel per kelas.
Data Train Seimbang: 17584 sampel tersimpan di /kaggle/working/data/qwen35_train.json


In [2]:
import os

%cd /kaggle/working

!rm -rf LLaMA-Factory

# 1. Hapus secara menyeluruh semua pustaka inti yang memicu konflik hierarki
!pip uninstall -y transformers accelerate peft bitsandbytes trl vllm llamafactory 2>/dev/null || true

# 2. Unduh kembali repositori resmi LLaMA-Factory dari sumber utama
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd /kaggle/working/LLaMA-Factory

# 3. Instal pustaka akselerasi dasar dan utilitas pemrosesan visual Qwen
!pip install -q qwen-vl-utils bitsandbytes

# 4. Eksekusi instalasi otomatis agar setup.py mencari versi ekuilibrium terbaiknya sendiri
!pip install -q -e .

/kaggle/working
Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 27409, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 27409 (delta 105), reused 30 (delta 30), pack-reused 27230 (from 4)
Receiving objects: 100% (27409/27409), 13.25 MiB | 25.31 MiB/s, done.
Resolving deltas: 100% (19627/19627), done.
/kaggle/working/LLaMA-Factory
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 56.1 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Check

In [9]:
import os
# Mencabut PyTorch versi 2.9 yang bermasalah beserta dependensinya
!pip uninstall -y torch torchvision torchaudio

# Memasang versi 2.5.1 yang sangat stabil dan kompatibel dengan CUDA 12.1 di Kaggle
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 100.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 89.9 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 72.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 93.8 MB/s eta 0:00:0000:0100:01
    

In [10]:
import json
import os
import shutil

DATA_DIR = '/kaggle/working/data'
LF_DATA_DIR = '/kaggle/working/LLaMA-Factory/data'

for f in os.listdir(DATA_DIR):
    if f.endswith('.json'):
        shutil.copy(os.path.join(DATA_DIR, f), LF_DATA_DIR)

info_path = os.path.join(LF_DATA_DIR, 'dataset_info.json')

dataset_info = {
    "adas_qwen_train": {
        "file_name": "qwen35_train.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"}
    },
    "adas_qwen_val": {
        "file_name": "qwen35_val.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"}
    },
    "adas_qwen_test": {
        "file_name": "qwen35_test.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"}
    }
}

if os.path.exists(info_path):
    with open(info_path, 'r', encoding='utf-8') as f:
        existing = json.load(f)
    existing.update(dataset_info)
    dataset_info = existing

with open(info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=2)

print("Dataset 4-Class berhasil diregistrasi ke sistem LLaMA-Factory.")

Dataset 4-Class berhasil diregistrasi ke sistem LLaMA-Factory.


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!llamafactory-cli train \
  --stage sft \
  --do_train True \
  --do_eval True \
  --model_name_or_path Qwen/Qwen3.5-4B \
  --trust_remote_code True \
  --dataset adas_qwen_train \
  --eval_dataset adas_qwen_val \
  --dataset_dir data \
  --template qwen3_5 \
  --finetuning_type lora \
  --lora_target all \
  --lora_rank 8 \
  --lora_alpha 16 \
  --lora_dropout 0.05 \
  --quantization_bit 4 \
  --bf16 True \
  --cutoff_len 1024 \
  --num_train_epochs 5 \
  --per_device_train_batch_size 1 \
  --per_device_eval_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --lr_scheduler_type cosine \
  --learning_rate 5e-5 \
  --weight_decay 0.05 \
  --warmup_ratio 0.1 \
  --logging_steps 10 \
  --eval_strategy steps \
  --eval_steps 500 \
  --save_strategy steps \
  --save_steps 500 \
  --load_best_model_at_end True \
  --metric_for_best_model eval_loss \
  --early_stopping_steps 3 \
  --output_dir /kaggle/working/qwen35_4b_lora \
  --overwrite_cache True \
  --overwrite_output_dir False \
  --report_to none

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[WARNING|2026-06-03 13:47:22] llamafactory.hparams.parser:149 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-06-03 13:47:22] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:771] 2026-06-03 13:47:22,300 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json
[INFO|configuration_utils.py:847] 2026-06-03 13:47:22,319 >> Model config Qwen3_5Config {
  "architectures": [
    "Qwen3_5ForConditionalGeneration"
  ],
  "image_token_id": 248056,
  "model_type": "qwen3_5",
  "text_config": {
    "attention_bias": false,
    "attention_dropout": 0.0,
    "attn_output_gate": true,
    "bos_token_id": null,
    "dtype": "bfloat16",
    "e

In [ ]:
%%bash
cd /kaggle/working/LLaMA-Factory

BEST_CKPT=$(ls -td /kaggle/working/qwen35_4b_lora/checkpoint-* 2>/dev/null | head -1)
echo "Menggunakan checkpoint terbaik: $BEST_CKPT"

llamafactory-cli train \
  --stage             sft \
  --do_predict        True \
  \
  --model_name_or_path    Qwen/Qwen3.5-4B \
  --adapter_name_or_path  $BEST_CKPT \
  --trust_remote_code     True \
  \
  --dataset             adas_qwen_test \
  --dataset_dir         data \
  --template            qwen3_5 \
  \
  --finetuning_type     lora \
  --quantization_bit    4 \
  --bf16                True \
  \
  --per_device_eval_batch_size  1 \
  --cutoff_len          256 \
  \
  --output_dir          /kaggle/working/qwen35_4b_test_results \
  --report_to           none \
  --overwrite_output_dir True